In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolTransforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from scipy.spatial.distance import mahalanobis, cdist
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = ["Arial"]
plt.rcParams["font.size"] = 12
plt.rcParams["axes.linewidth"] = 1.2
plt.rcParams["xtick.major.width"] = 1.2
plt.rcParams["ytick.major.width"] = 1.2
plt.rcParams["figure.dpi"] = 300

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DIM_F1 = 20
DIM_F2_RAW = 1024
DIM_F2_PCA = 128
DIM_F3 = 10
DIM_F4 = 43
DIM_HB = 2
DIM_ONE_ASSAY = 3
DIM_INTER = 14

MODAL_DIMS_SINGLE = [DIM_F1, DIM_F2_PCA, DIM_F3, DIM_F4, DIM_HB, DIM_ONE_ASSAY]
FUSION_HIDDEN = 192
HA_HB_DIM = 192
GAN_INPUT_DIM = HA_HB_DIM * 2 + DIM_INTER

EMBED_DIM = 32
EXP_DIM = 32
INPUT_DIM = GAN_INPUT_DIM + EMBED_DIM + EXP_DIM

ASSAY_COLS = [
    "SMILES1-assay1", "SMILES1-assay2", "SMILES1-assay3",
    "SMILES2-assay1", "SMILES2-assay2", "SMILES2-assay3"
]

BATCH_SIZE = 32
EPOCHS = 150
LR = 2e-4
PATIENCE = 30
NUM_CLASSES = 5
WEIGHT_DECAY = 1e-5

GAN_NOISE_DIM = 128
GAN_EPOCHS = 200
GAN_BATCH_DEFAULT = 16
GAN_LR = 1e-4
GP_LAMBDA = 15
CRITIC_ITER = 8

cls_count = {0:929, 1:870, 2:897, 3:260, 4:195}
target_num = 929
need_gen = {c: target_num - cnt for c, cnt in cls_count.items()}
print("每类需要生成样本数：", need_gen)

MD_THRESH_RATIO = 0.98
KNN_THRESH_RATIO = 0.98
TANIMOTO_CUTOFF = 0.50
K_NEIGHBOR = 5
AD_JUDGE_MODE = "loose"

class ModelApplicabilityDomain:
    def __init__(self, train_X_scaled, train_fpA, train_fpB):
        self.X_train = train_X_scaled
        self.fpA_train = train_fpA
        self.fpB_train = train_fpB
        self.n_train = self.X_train.shape[0]

        self.X_mean = np.mean(self.X_train, axis=0)
        self.X_cov = np.cov(self.X_train.T)
        self.X_cov_inv = np.linalg.pinv(self.X_cov)
        train_md = np.array([mahalanobis(x, self.X_mean, self.X_cov_inv) for x in self.X_train])
        self.md_threshold = np.quantile(train_md, MD_THRESH_RATIO)
        self.train_md = train_md

        knn_dist = cdist(self.X_train, self.X_train, metric="euclidean")
        np.fill_diagonal(knn_dist, np.inf)
        knn_sort = np.sort(knn_dist, axis=1)
        avg_k_dist = np.mean(knn_sort[:, :K_NEIGHBOR], axis=1)
        self.knn_threshold = np.quantile(avg_k_dist, KNN_THRESH_RATIO)
        self.train_knn = avg_k_dist

    def calc_min_tanimoto(self, fp_new, fp_train_arr):
        fp_new_bin = fp_new.astype(bool)
        t_list = []
        for fp_t in fp_train_arr:
            fp_t_bin = fp_t.astype(bool)
            inter = np.logical_and(fp_new_bin, fp_t_bin).sum()
            union = np.logical_or(fp_new_bin, fp_t_bin).sum()
            t = inter / union if union > 1e-8 else 0.0
            t_list.append(t)
        t_list.sort(reverse=True)
        top5 = t_list[:5]
        return np.mean(top5)

    def judge_sample(self, x_scaled, fpA_new, fpB_new):
        md = mahalanobis(x_scaled, self.X_mean, self.X_cov_inv)
        md_in = md < self.md_threshold

        dist_all = cdist(x_scaled.reshape(1, -1), self.X_train, metric="euclidean")[0]
        dist_sort = np.sort(dist_all)
        avg_knn = np.mean(dist_sort[:K_NEIGHBOR])
        knn_in = avg_knn < self.knn_threshold

        tA = self.calc_min_tanimoto(fpA_new, self.fpA_train)
        tB = self.calc_min_tanimoto(fpB_new, self.fpB_train)
        fp_in = (tA >= TANIMOTO_CUTOFF) or (tB >= TANIMOTO_CUTOFF)

        if AD_JUDGE_MODE == "strict":
            in_domain = md_in and knn_in and fp_in
        elif AD_JUDGE_MODE == "loose":
            in_domain = (md_in and knn_in) or fp_in
        else:
            raise ValueError("AD_JUDGE_MODE only support strict / loose")

        return {
            "md": md,
            "md_thresh": self.md_threshold,
            "knn_avg": avg_knn,
            "knn_thresh": self.knn_threshold,
            "min_tani_A": tA,
            "min_tani_B": tB,
            "in_domain": in_domain
        }

def plot_ad_figure(ad_fusion, train_md_fusion, train_knn_fusion, test_df):
    required_cols = [
        "md", "knn_avg", "min_tani_A", "min_tani_B", "in_domain"
    ]
    for col in required_cols:
        if col not in test_df.columns:
            raise ValueError(f"DataFrame缺失关键AD字段：{col}")

    fig1, ax1 = plt.subplots(figsize=(8,6))
    ax1.scatter(train_md_fusion, train_knn_fusion, c="#cccccc", alpha=0.4, s=12, label="Training set (fusion feat)")
    test_in = test_df[test_df["in_domain"]==True]
    test_out = test_df[test_df["in_domain"]==False]
    if len(test_in) > 0:
        ax1.scatter(test_in["md"], test_in["knn_avg"], c="#2E86AB", s=22, alpha=0.7, label="Test set (in-domain)")
    if len(test_out) > 0:
        ax1.scatter(test_out["md"], test_out["knn_avg"], c="#E63946", s=22, alpha=0.7, label="Test set (out-of-domain)")
    ax1.axvline(x=ad_fusion.md_threshold, color="black", linestyle="--", lw=1.3, label=f"{MD_THRESH_RATIO} MD threshold")
    ax1.axhline(y=ad_fusion.knn_threshold, color="black", linestyle=":", lw=1.3, label=f"{K_NEIGHBOR}-NN threshold")
    ax1.set_xlabel("Mahalanobis distance (Fusion feature)")
    ax1.set_ylabel(f"Average {K_NEIGHBOR}-NN Euclidean distance (Fusion feature)")
    ax1.legend(frameon=True, loc="upper right")
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig("AD_Fusion_MD_vs_KNN.tif", dpi=300, bbox_inches="tight")
    plt.savefig("AD_Fusion_MD_vs_KNN.pdf", bbox_inches="tight")
    plt.show()

    fig2, (ax2a, ax2b) = plt.subplots(1,2, figsize=(10,4))
    sns.histplot(test_df["min_tani_A"], bins=30, ax=ax2a, color="#457B9D", edgecolor="white")
    ax2a.axvline(x=TANIMOTO_CUTOFF, c="red", ls="--", lw=1.2)
    ax2a.set_xlabel("Minimum Tanimoto similarity (SMILES1)")
    ax2a.set_ylabel("Count")
    ax2a.spines["top"].set_visible(False)
    ax2a.spines["right"].set_visible(False)
    sns.histplot(test_df["min_tani_B"], bins=30, ax=ax2b, color="#1D3557", edgecolor="white")
    ax2b.axvline(x=TANIMOTO_CUTOFF, c="red", ls="--", lw=1.2)
    ax2b.set_xlabel("Minimum Tanimoto similarity (SMILES2)")
    ax2b.set_ylabel("Count")
    ax2b.spines["top"].set_visible(False)
    ax2b.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig("AD_Tanimoto_Hist.tif", dpi=300, bbox_inches="tight")
    plt.savefig("AD_Tanimoto_Hist.pdf", bbox_inches="tight")
    plt.show()

    test_df.to_csv("AD_statistics.csv", index=False, encoding="utf-8-sig")
    print("===== AD图表保存完成 =====")
    print("融合AD主图: AD_Fusion_MD_vs_KNN")
    print("Tanimoto直方图 & AD指标表: AD_statistics.csv")

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = torch.tensor(alpha, dtype=torch.float32).to(device) if alpha else None

    def forward(self, inputs, targets):
        logpt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(logpt)
        logpt = (1 - pt) ** self.gamma * logpt
        if self.alpha is not None:
            logpt = self.alpha * logpt
        loss = F.nll_loss(logpt, targets, reduction=self.reduction)
        return loss

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim, lambda_c=0.001):
        super().__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.lambda_c = lambda_c
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim).to(device))

    def forward(self, feat, label):
        batch_size = feat.size(0)
        feat = feat.view(batch_size, -1)
        centers_batch = self.centers.index_select(0, label)
        dist = torch.sum((feat - centers_batch) ** 2, dim=1)
        return self.lambda_c * torch.mean(dist)

def to_fixed_len(arr, target_len):
    arr = np.asarray(arr, dtype=np.float32).ravel()
    if len(arr) < target_len:
        arr = np.concatenate([arr, np.zeros(target_len - len(arr), dtype=np.float32)])
    else:
        arr = arr[:target_len]
    return arr

def generate_3d_conformer(mol, max_attempts=10):
    try:
        mol_3d = Chem.AddHs(mol)
        for _ in range(max_attempts):
            if AllChem.EmbedMolecule(mol_3d, randomSeed=42) == 0:
                AllChem.MMFFOptimizeMolecule(mol_3d)
                return mol_3d
        return None
    except:
        return None

def get_3d_geometric_features(mol_3d):
    feats = []
    if mol_3d is not None and mol_3d.GetNumConformers() > 0:
        try:
            conf = mol_3d.GetConformer()
            num_atoms = mol_3d.GetNumAtoms()
            coords = np.array([[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
                               for i in range(num_atoms)])
            feats.extend(coords.mean(axis=0).tolist())
            feats.extend(coords.std(axis=0).tolist())
            feats.extend(coords.max(axis=0).tolist())
            feats.extend(coords.min(axis=0).tolist())

            bond_lengths = [rdMolTransforms.GetBondLength(conf, b.GetBeginAtomIdx(), b.GetEndAtomIdx())
                            for b in mol_3d.GetBonds()]
            feats += [np.mean(bond_lengths), np.std(bond_lengths), np.max(bond_lengths), np.min(bond_lengths)] if bond_lengths else [0.0]*4

            angles = []
            for a in range(num_atoms):
                for n in mol_3d.GetAtomWithIdx(a).GetNeighbors():
                    angles.append(rdMolTransforms.GetAngleDeg(conf, a, n.GetIdx(), a))
            feats += [np.mean(angles), np.std(angles), np.max(angles), np.min(angles)] if angles else [0.0]*4

            dihedrals = []
            for b in mol_3d.GetBonds():
                a1, a2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
                for n1 in mol_3d.GetAtomWithIdx(a1).GetNeighbors():
                    for n2 in mol_3d.GetAtomWithIdx(a2).GetNeighbors():
                        if n1.GetIdx() != a2 and n2.GetIdx() != a1:
                            dihedrals.append(rdMolTransforms.GetDihedralDeg(conf, n1.GetIdx(), a1, a2, n2.GetIdx()))
            feats += dihedrals[:20]
        except:
            pass
    return to_fixed_len(feats, DIM_F4)

def extract_mol_features_raw(smi):
    f1 = np.zeros(DIM_F1, dtype=np.float32)
    f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f3 = np.zeros(DIM_F3, dtype=np.float32)
    f4 = np.zeros(DIM_F4, dtype=np.float32)
    hb = np.zeros(DIM_HB, dtype=np.float32)

    if pd.isna(smi) or not isinstance(smi, str) or len(smi.strip()) == 0:
        return f1, f2_raw, f3, f4, hb

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return f1, f2_raw, f3, f4, hb

    f1 = np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol), Descriptors.FractionCSP3(mol),
        Descriptors.NumValenceElectrons(mol), Descriptors.HeavyAtomCount(mol),
        Descriptors.NHOHCount(mol), Descriptors.NOCount(mol),
        Descriptors.NumSaturatedRings(mol), Descriptors.NumHeteroatoms(mol),
        Descriptors.NumAliphaticRings(mol),
        Descriptors.MaxPartialCharge(mol) if Descriptors.MaxPartialCharge(mol) else 0.0,
        Descriptors.MinPartialCharge(mol) if Descriptors.MinPartialCharge(mol) else 0.0,
        Descriptors.MolMR(mol), 0.0, 0.0,
    ], dtype=np.float32)
    f1 = to_fixed_len(f1, DIM_F1)

    try:
        fp_bit = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=DIM_F2_RAW)
        f2_raw = np.array(fp_bit, dtype=np.float32)
    except:
        f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f2_raw = to_fixed_len(f2_raw, DIM_F2_RAW)

    f3 = np.array([
        Descriptors.MolWt(mol) * 0.1, Descriptors.MolLogP(mol) * 0.5, Descriptors.TPSA(mol) * 0.01,
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
        Descriptors.NumHeteroatoms(mol), Descriptors.NumRotatableBonds(mol),
        Descriptors.LabuteASA(mol) if Descriptors.LabuteASA(mol) else 0.0,
    ], dtype=np.float32)
    f3 = to_fixed_len(f3, DIM_F3)

    mol_3d = generate_3d_conformer(mol)
    f4 = get_3d_geometric_features(mol_3d)

    h_don = Descriptors.NumHDonors(mol)
    h_acc = Descriptors.NumHAcceptors(mol)
    hb = np.array([h_don, h_acc], dtype=np.float32)
    hb = to_fixed_len(hb, DIM_HB)

    return f1, f2_raw, f3, f4, hb

def calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b):
    def cos_sim(u, v):
        n1, n2 = np.linalg.norm(u), np.linalg.norm(v)
        return 0.0 if (n1 < 1e-8 or n2 < 1e-8) else np.dot(u, v) / (n1 * n2)
    def l2_dist(u, v):
        return np.linalg.norm(u - v)

    cos1 = cos_sim(f1a, f1b)
    dist1 = l2_dist(f1a, f1b)
    had1 = (f1a * f1b).mean()

    cos3 = cos_sim(f4a, f4b)
    dist3 = l2_dist(f4a, f4b)
    had3 = (f4a * f4b).mean()

    had_low = (f3a * f3b).mean()

    da, aa = hb_a[0], hb_a[1]
    db, ab = hb_b[0], hb_b[1]
    sum_d, sum_a = da + db, aa + ab
    diff_d, diff_a = abs(da - db), abs(aa - ab)
    mul_d, mul_a = da * db, aa * ab

    inter = np.array([
        cos1, dist1, had1, cos3, dist3, had3, had_low,
        sum_d, sum_a, diff_d, diff_a, mul_d, mul_a, 0.0
    ], dtype=np.float32)
    return to_fixed_len(inter, DIM_INTER)

class CrossInteractAttnFusion(nn.Module):
    def __init__(self, modal_dims, hidden_dim, out_dim, head=4):
        super().__init__()
        self.modal_dims = modal_dims
        self.num_modal = len(modal_dims)
        self.head = head
        self.head_dim = hidden_dim // head
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim

        self.projs = nn.ModuleList([nn.Linear(d, hidden_dim) for d in modal_dims])
        self.ln_proj = nn.LayerNorm(hidden_dim)
        self.qkv_global = nn.Linear(hidden_dim, hidden_dim * 3)
        self.assay_gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Sigmoid()
        )
        self.cross_q = nn.Linear(hidden_dim, hidden_dim)
        self.cross_k = nn.Linear(hidden_dim, hidden_dim)
        self.cross_v = nn.Linear(hidden_dim, hidden_dim)
        self.ln_cross = nn.LayerNorm(hidden_dim)
        self.concat_ln = nn.LayerNorm(hidden_dim * self.num_modal)
        self.out_linear = nn.Linear(hidden_dim * self.num_modal, out_dim)
        self.drop = nn.Dropout(0.35)

    def forward(self, modal_list):
        bs = modal_list[0].shape[0]
        hidden_list = []
        for idx, m in enumerate(modal_list):
            h = self.projs[idx](m)
            h = self.ln_proj(h)
            hidden_list.append(h)
        stack = torch.stack(hidden_list, dim=1)
        qkv = self.qkv_global(stack).reshape(bs, self.num_modal, 3, self.head, self.head_dim).permute(2,0,3,1,4)
        q_g, k_g, v_g = qkv[0], qkv[1], qkv[2]
        attn_score = torch.matmul(q_g, k_g.transpose(-1,-2)) / np.sqrt(self.head_dim)
        attn_weight = F.softmax(attn_score, dim=-1)
        global_attn_out = torch.matmul(attn_weight, v_g).transpose(1,2).reshape(bs, self.num_modal, self.hidden_dim)
        assay_feat = stack[:, -1, :]
        gate_w = self.assay_gate(assay_feat).unsqueeze(1)
        gated_feat = global_attn_out * gate_w
        molA_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        molB_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        q_c = self.cross_q(molA_feat)
        k_c = self.cross_k(molB_feat)
        v_c = self.cross_v(molB_feat)
        cross_attn = F.softmax(torch.matmul(q_c, k_c.transpose(-1,-2)) / np.sqrt(self.head_dim), dim=-1)
        cross_out = torch.matmul(cross_attn, v_c)
        cross_out = self.ln_cross(cross_out)
        fused_all = gated_feat + cross_out
        concat = fused_all.reshape(bs, -1)
        concat = self.concat_ln(concat)
        out = self.drop(self.out_linear(concat))
        return out

class ToxicityDataset(Dataset):
    def __init__(self, feat, species, exposure, label):
        self.feat = feat
        self.species = species
        self.exposure = exposure
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.feat[idx]).float(),
            torch.tensor(self.species[idx], dtype=torch.long),
            torch.tensor([self.exposure[idx]], dtype=torch.float32),
            torch.tensor(self.label[idx], dtype=torch.long)
        )

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.4):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        residual = x
        out = self.dropout(self.act(self.bn1(self.fc1(x))))
        out = self.bn2(self.fc2(out))
        out += residual
        return self.act(out)

class ImprovedToxicModel(nn.Module):
    def __init__(self, total_feat_dim, embed_dim, exp_dim, species_num, num_classes):
        super().__init__()
        self.embed_dim = embed_dim
        self.exp_dim = exp_dim
        self.total_feat_dim = total_feat_dim
        self.species_emb = nn.Embedding(species_num, embed_dim)
        self.exp_proj = nn.Linear(1, exp_dim)
        self.backbone = nn.Sequential(
            nn.Linear(INPUT_DIM, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(1024),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(512),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            ResBlock(256),
        )
        self.classifier = nn.Linear(256, num_classes)
        self.feature_out_dim = 256

    def forward(self, full_feat, species, exposure):
        sp_feat = self.species_emb(species)
        exp_feat = F.gelu(self.exp_proj(exposure)).squeeze(1)
        all_feat = torch.cat([full_feat, sp_feat, exp_feat], dim=-1)
        hidden = self.backbone(all_feat)
        logits = self.classifier(hidden)
        return logits, hidden

class WGANGP_Generator(nn.Module):
    def __init__(self, noise_dim, cond_dim, feat_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 1024),
            nn.LayerNorm(1024),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.25),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.25),
            nn.Linear(512, feat_dim),
        )
    def forward(self, z, cond):
        return self.net(torch.cat([z, cond], dim=1))

class WGANGP_Critic(nn.Module):
    def __init__(self, feat_dim, cond_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim + cond_dim, 512),
            nn.LeakyReLU(0.2, True),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.LeakyReLU(0.2, True),
            nn.Linear(256, 1),
        )
    def forward(self, feat, cond):
        return self.net(torch.cat([feat, cond], dim=1))

def compute_gp(critic, real, fake, cond):
    bs = real.shape[0]
    alpha = torch.rand(bs, 1).to(device)
    interp = (alpha * real + (1 - alpha) * fake).clone().detach()
    interp.requires_grad = True
    out = critic(interp, cond)
    grad = torch.autograd.grad(outputs=out.sum(), inputs=interp, create_graph=True)[0]
    grad_norm = grad.norm(2, dim=1)
    return torch.mean((grad_norm - 1) ** 2)

def train_wgangp(X, y, num_classes, noise_dim, epochs, batch_size_max, lr):
    feat_dim = X.shape[1]
    sample_num = X.shape[0]
    batch_size = min(batch_size_max, sample_num)
    y_onehot = np.eye(num_classes)[y]
    X_tensor = torch.from_numpy(X).float().to(device)
    y_tensor = torch.from_numpy(y_onehot).float().to(device)
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    G = WGANGP_Generator(noise_dim, num_classes, feat_dim).to(device)
    C = WGANGP_Critic(feat_dim, num_classes).to(device)
    opt_G = Adam(G.parameters(), lr=lr, betas=(0.0,0.9))
    opt_C = Adam(C.parameters(), lr=lr, betas=(0.0,0.9))
    print("开始训练当前类 WGAN-GP（强化模式防崩溃配置）...")
    for epoch in range(epochs):
        G.train()
        C.train()
        total_c_loss = 0.0
        total_g_loss = 0.0
        for real_feat, real_cond in dataloader:
            bsz = real_feat.size(0)
            for _ in range(CRITIC_ITER):
                opt_C.zero_grad()
                real_out = C(real_feat, real_cond)
                z = torch.randn(bsz, noise_dim, device=device)
                fake_feat = G(z, real_cond)
                fake_out = C(fake_feat.detach(), real_cond)
                gp = compute_gp(C, real_feat, fake_feat, real_cond)
                c_loss = torch.mean(fake_out) - torch.mean(real_out) + GP_LAMBDA * gp
                c_loss.backward()
                opt_C.step()
            total_c_loss += c_loss.item()
            opt_G.zero_grad()
            z = torch.randn(bsz, noise_dim, device=device)
            fake_feat = G(z, real_cond)
            fake_out = C(fake_feat, real_cond)
            g_loss = -torch.mean(fake_out)
            g_loss.backward()
            opt_G.step()
            total_g_loss += g_loss.item()
        total_c_loss /= len(dataloader) * CRITIC_ITER
        total_g_loss /= len(dataloader)
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:2d} | Critic Loss: {total_c_loss:.4f} | Gen Loss: {total_g_loss:.4f}")
    return G

def generate_by_class(G, cls, gen_num, num_classes, noise_dim):
    G.eval()
    cond = torch.zeros(gen_num, num_classes, device=device)
    cond[:, cls] = 1.0
    z = torch.randn(gen_num, noise_dim, device=device)
    with torch.no_grad():
        fake_feat = G(z, cond)
        noise = torch.normal(mean=0.0, std=0.01, size=fake_feat.shape, device=device)
        fake_feat = fake_feat + noise
    return fake_feat.cpu().numpy(), np.full(gen_num, cls)

if __name__ == "__main__":
    FILE_PATH = "reproduction-toxicity-traindata-Enumerator.xlsx" #neurotoxicity/immunotoxicity/hepatotoxicity/developmental-toxicity-traindata-Enumerator.xlsx
    df = pd.read_excel(FILE_PATH, sheet_name=0)
    df["Interaction types of compounds"] = df["Interaction types of compounds"].str.strip()
    label_map = {
        "Antagonistic effect": 0,
        "Synergistic effect": 1,
        "Not mentioned": 2,
        "Additive effect": 3,
        "Independent effect": 4
    }
    df = df[df["Interaction types of compounds"].isin(label_map.keys())].reset_index(drop=True)
    df["label"] = df["Interaction types of compounds"].map(label_map)
    print("原始标签分布：")
    print(df["Interaction types of compounds"].value_counts())

    df["species"] = df["species"].fillna("Unknown")
    sp_enc = LabelEncoder()
    df["sp_enc"] = sp_enc.fit_transform(df["species"])
    num_sp = len(sp_enc.classes_)

    print("\n收集指纹并训练PCA...")
    all_fp_raw = []
    for _, row in df.iterrows():
        _, f2a_raw, _, _, _ = extract_mol_features_raw(row["SMILES1"])
        _, f2b_raw, _, _, _ = extract_mol_features_raw(row["SMILES2"])
        all_fp_raw.append(f2a_raw)
        all_fp_raw.append(f2b_raw)
    all_fp_raw = np.array(all_fp_raw, dtype=np.float32)
    pca = PCA(n_components=DIM_F2_PCA, random_state=42)
    pca.fit(all_fp_raw)
    print(f"PCA累计方差解释率: {np.sum(pca.explained_variance_ratio_):.4f}")

    fusion_model = CrossInteractAttnFusion(MODAL_DIMS_SINGLE, FUSION_HIDDEN, HA_HB_DIM).to(device)
    fusion_model.eval()

    print(f"\n单分子融合输出HA/HB维度: {HA_HB_DIM}")
    print(f"GAN输入基础特征维度(HA+HB+inter): {GAN_INPUT_DIM}")
    X_list = []
    sp_list = []
    exp_list = []
    lab_list = []
    train_fpA_all = []
    train_fpB_all = []
    df_full_feat_cache = []

    for idx, row in df.iterrows():
        f1a, f2a_raw, f3a, f4a, hb_a = extract_mol_features_raw(row["SMILES1"])
        f1b, f2b_raw, f3b, f4b, hb_b = extract_mol_features_raw(row["SMILES2"])
        f2a = pca.transform(f2a_raw.reshape(1, -1))[0]
        f2b = pca.transform(f2b_raw.reshape(1, -1))[0]
        assayA = row[["SMILES1-assay1","SMILES1-assay2","SMILES1-assay3"]].values.astype(np.float32)
        assayB = row[["SMILES2-assay1","SMILES2-assay2","SMILES2-assay3"]].values.astype(np.float32)
        inter_feat = calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b)
        modA = [
            torch.from_numpy(f1a).float().unsqueeze(0).to(device),
            torch.from_numpy(f2a).float().unsqueeze(0).to(device),
            torch.from_numpy(f3a).float().unsqueeze(0).to(device),
            torch.from_numpy(f4a).float().unsqueeze(0).to(device),
            torch.from_numpy(hb_a).float().unsqueeze(0).to(device),
            torch.from_numpy(assayA).float().unsqueeze(0).to(device)
        ]
        modB = [
            torch.from_numpy(f1b).float().unsqueeze(0).to(device),
            torch.from_numpy(f2b).float().unsqueeze(0).to(device),
            torch.from_numpy(f3b).float().unsqueeze(0).to(device),
            torch.from_numpy(f4b).float().unsqueeze(0).to(device),
            torch.from_numpy(hb_b).float().unsqueeze(0).to(device),
            torch.from_numpy(assayB).float().unsqueeze(0).to(device)
        ]
        with torch.no_grad():
            HA = fusion_model(modA).squeeze(0).cpu().numpy()
            HB = fusion_model(modB).squeeze(0).cpu().numpy()
        fusion_feat = np.concatenate([HA, HB, inter_feat]).reshape(1, -1)
        fusion_feat_scaled = scaler.transform(fusion_feat)[0]
        train_fpA_all.append(f2a_raw)
        train_fpB_all.append(f2b_raw)
        df_full_feat_cache.append({
            "original_idx": idx,
            "fusion_feat": fusion_feat,
            "fpA_raw": f2a_raw,
            "fpB_raw": f2b_raw
        })

    train_fpA_all = np.array(train_fpA_all)
    train_fpB_all = np.array(train_fpB_all)
    X_raw = np.stack(X_list, axis=0)
    sp_arr = np.array(sp_list)
    exp_arr = np.array(exp_list, dtype=np.float32).reshape(-1, 1)
    y_arr = np.array(lab_list)
    print(f"融合后GAN基础特征矩阵shape: {X_raw.shape}")

    feat_imp = SimpleImputer(strategy="mean")
    X_raw = feat_imp.fit_transform(X_raw)
    exp_imp = SimpleImputer(strategy="mean")
    exp_arr = exp_imp.fit_transform(exp_arr)
    scaler = StandardScaler()
    X_raw = scaler.fit_transform(X_raw)
    exp_scaler = StandardScaler()
    exp_arr = exp_scaler.fit_transform(exp_arr)

    X_train, X_test, sp_train, sp_test, exp_train, exp_test, y_train, y_test, train_idx, test_idx = \
        train_test_split(X_raw, sp_arr, exp_arr, y_arr, df.index, test_size=0.2,
                         random_state=42, stratify=y_arr)

    ad_fusion = ModelApplicabilityDomain(X_train, train_fpA_all, train_fpB_all)
    print(f"\n===== AD宽松配置：loose + MD/KNN分位数0.98 + Tanimoto截断0.60 =====")
    print(f"融合AD MD阈值:{ad_fusion.md_threshold:.4f} | 5NN阈值:{ad_fusion.knn_threshold:.4f}")
    print(f"判定模式: {AD_JUDGE_MODE}")

    test_ad_records = []
    test_in_flag = []
    match_success_cnt = 0
    match_fail_cnt = 0

    for test_idx_pos, original_idx in enumerate(test_idx):
        match_item = df_full_feat_cache[original_idx]
        feat_t = X_test[test_idx_pos]
        if match_item is not None:
            res = ad_fusion.judge_sample(
                x_scaled=feat_t,
                fpA_new=match_item["fpA_raw"],
                fpB_new=match_item["fpB_raw"]
            )
            test_ad_records.append(res)
            test_in_flag.append(res["in_domain"])
            match_success_cnt += 1
        else:
            empty_rec = {
                "md": np.nan, "md_thresh": ad_fusion.md_threshold, "knn_avg": np.nan, "knn_thresh": ad_fusion.knn_threshold,
                "min_tani_A": np.nan, "min_tani_B": np.nan, "in_domain": False
            }
            test_ad_records.append(empty_rec)
            test_in_flag.append(False)
            match_fail_cnt += 1

    test_in_flag = np.array(test_in_flag)
    print(f"\n===== 匹配结果统计 =====")
    print(f"匹配成功样本数: {match_success_cnt} | 匹配失败样本数: {match_fail_cnt}")
    print(f"融合AD 总测试样本{len(test_in_flag)} | 域内{np.sum(test_in_flag)} | 域外{len(test_in_flag)-np.sum(test_in_flag)}")

    test_df = pd.DataFrame(test_ad_records)
    test_df = test_df.dropna(how="all")
    plot_ad_figure(
        ad_fusion=ad_fusion,
        train_md_fusion=ad_fusion.train_md,
        train_knn_fusion=ad_fusion.train_knn,
        test_df=test_df
    )

    print("\n===== 开始按类别训练WGAN-GP扩充少数类特征 =====")
    gen_X_total = []
    gen_y_total = []
    for cls in range(NUM_CLASSES):
        target_gen = need_gen[cls]
        if target_gen <= 0:
            print(f"类别{cls}样本充足，无需生成")
            continue
        cls_mask = y_train == cls
        X_cls = X_train[cls_mask]
        y_cls = y_train[cls_mask]
        print(f"\n类别{cls}：原始样本{len(X_cls)}，需生成{target_gen}条")
        G_model = train_wgangp(X_cls, y_cls, NUM_CLASSES, GAN_NOISE_DIM, GAN_EPOCHS, GAN_BATCH_DEFAULT, GAN_LR)
        gen_feat, gen_label = generate_by_class(G_model, cls, target_gen, NUM_CLASSES, GAN_NOISE_DIM)
        gen_X_total.append(gen_feat)
        gen_y_total.append(gen_label)
    if len(gen_X_total) > 0:
        gen_X_stack = np.vstack(gen_X_total)
        gen_y_stack = np.hstack(gen_y_total)
        X_train_bal = np.concatenate([X_train, gen_X_stack], axis=0)
        y_train_bal = np.concatenate([y_train, gen_y_stack], axis=0)
        sp_train_bal = np.concatenate([sp_train, np.random.choice(sp_train, size=len(gen_y_stack), replace=True)], axis=0)
        exp_train_bal = np.concatenate([exp_train, np.random.choice(exp_train.reshape(-1), size=len(gen_y_stack), replace=True).reshape(-1,1)], axis=0)
    else:
        X_train_bal = X_train
        y_train_bal = y_train
        sp_train_bal = sp_train
        exp_train_bal = exp_train
    print(f"\n均衡后训练集总样本量：{len(X_train_bal)}")
    unique_cls, cls_counts = np.unique(y_train_bal, return_counts=True)
    for c, cnt in zip(unique_cls, cls_counts):
        print(f"类别{c} 样本数：{cnt}")

    train_dataset = ToxicityDataset(X_train_bal, sp_train_bal, exp_train_bal, y_train_bal)
    test_dataset = ToxicityDataset(X_test, sp_test, exp_test, y_test)
    class_weight_arr = np.array([len(y_train_bal) / cls_counts[c] for c in range(NUM_CLASSES)])
    sample_weights = np.array([class_weight_arr[label] for label in y_train_bal])
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = ImprovedToxicModel(
        total_feat_dim=GAN_INPUT_DIM,
        embed_dim=EMBED_DIM,
        exp_dim=EXP_DIM,
        species_num=num_sp,
        num_classes=NUM_CLASSES
    ).to(device)
    focal_alpha = [len(y_train_bal) / cls_counts[c] for c in range(NUM_CLASSES)]
    loss_cls = FocalLoss(alpha=focal_alpha, gamma=2.0)
    loss_center = CenterLoss(num_classes=NUM_CLASSES, feat_dim=256, lambda_c=0.001)
    optimizer = Adam(
        list(model.parameters()) + list(loss_center.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    best_acc = 0.0
    stop_count = 0
    print("\n===== 开始分类模型训练 =====")
    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0.0
        for batch_feat, batch_sp, batch_exp, batch_y in train_loader:
            batch_feat = batch_feat.to(device)
            batch_sp = batch_sp.to(device)
            batch_exp = batch_exp.to(device)
            batch_y = batch_y.to(device)
            logits, feat_hidden = model(batch_feat, batch_sp, batch_exp)
            loss1 = loss_cls(logits, batch_y)
            loss2 = loss_center(feat_hidden, batch_y)
            total_loss = loss1 + loss2
            optimizer.zero_grad()
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            total_train_loss += total_loss.item()
        avg_train_loss = total_train_loss / len(train_loader)
        scheduler.step()

        model.eval()
        test_pred_all = []
        test_true_all = []
        with torch.no_grad():
            for batch_feat, batch_sp, batch_exp, batch_y in test_loader:
                batch_feat = batch_feat.to(device)
                batch_sp = batch_sp.to(device)
                batch_exp = batch_exp.to(device)
                logits, _ = model(batch_feat, batch_sp, batch_exp)
                pred = torch.argmax(logits, dim=1).cpu().numpy()
                true = batch_y.cpu().numpy()
                test_pred_all.extend(pred)
                test_true_all.extend(true)
        test_acc = accuracy_score(test_true_all, test_pred_all)
        print(f"Epoch {epoch+1:3d} | Train Loss:{avg_train_loss:.4f} | Test Acc:{test_acc:.4f} | Best Acc:{best_acc:.4f}")

        if test_acc > best_acc:
            best_acc = test_acc
            stop_count = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "fusion_model": fusion_model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "best_acc": best_acc,
                "scaler": scaler,
                "exp_scaler": exp_scaler,
                "pca": pca,
                "sp_enc": sp_enc,
                "ad_fusion": ad_fusion
            }, "reproduction_modelAD.pth") #neurotoxicity/immunotoxicity/hepatotoxicity/developmental-toxicity_modelAD.pth
            print("✅ 保存最优模型 checkpoint")
        else:
            stop_count += 1
            if stop_count >= PATIENCE:
                print(f"早停触发，连续{PATIENCE}轮无提升，终止训练")
                break

    print("\n===== 加载最优模型，完整测试集评估 =====")
    ckpt = torch.load("reproduction_modelAD.pth", map_location=device) #neurotoxicity/immunotoxicity/hepatotoxicity/developmental-toxicity_modelAD.pth
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    preds_total = []
    trues_total = []
    with torch.no_grad():
        for feat, sp, exp, y in test_loader:
            feat = feat.to(device)
            sp = sp.to(device)
            exp = exp.to(device)
            logits, _ = model(feat, sp, exp)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            preds_total.extend(pred)
            trues_total.extend(y.numpy())
    test_acc_final = accuracy_score(trues_total, preds_total)
    print(f"全局最优测试准确率：{test_acc_final:.4f}")
    print("\n完整分类报告：")
    print(classification_report(trues_total, preds_total, target_names=list(label_map.keys()), digits=4))

    trues_arr = np.array(trues_total)
    preds_arr = np.array(preds_total)
    fusion_flag = test_in_flag

    print("\n========= 【唯一判定标准：融合特征AD（loose宽松规则）】分层模型性能 =========")
    fuse_in_true = trues_arr[fusion_flag]
    fuse_in_pred = preds_arr[fusion_flag]
    fuse_in_acc = accuracy_score(fuse_in_true, fuse_in_pred) if len(fuse_in_true) > 0 else np.nan
    print(f"融合AD 域内样本数：{len(fuse_in_true)}，域内准确率：{fuse_in_acc:.4f}")
    if len(fuse_in_true) > 0:
        print("融合AD域内分类报告：")
        print(classification_report(fuse_in_true, fuse_in_pred, labels=[0,1,2,3,4], target_names=list(label_map.keys()), digits=4, zero_division=0))
    fuse_out_true = trues_arr[~fusion_flag]
    fuse_out_pred = preds_arr[~fusion_flag]
    fuse_out_acc = accuracy_score(fuse_out_true, fuse_out_pred) if len(fuse_out_true) > 0 else np.nan
    print(f"\n融合AD 域外样本数：{len(fuse_out_true)}，域外准确率：{fuse_out_acc:.4f}")
    if len(fuse_out_true) > 0:
        print("融合AD域外分类报告：")
        print(classification_report(fuse_out_true, fuse_out_pred, labels=[0,1,2,3,4], target_names=list(label_map.keys()), digits=4, zero_division=0))

def predict_external_mixture(ckpt_path, smi1, smi2, a1,a2,a3,a4,a5,a6, species_name, exp_day):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device)
    scaler = ckpt["scaler"]
    exp_scaler = ckpt["exp_scaler"]
    pca = ckpt["pca"]
    sp_enc = ckpt["sp_enc"]
    ad_fusion = ckpt["ad_fusion"]
    fusion_model = CrossInteractAttnFusion(MODAL_DIMS_SINGLE, FUSION_HIDDEN, HA_HB_DIM).
    fusion_model.load_state_dict(ckpt["fusion_model"])
    fusion_model.eval()
    num_sp = len(sp_enc.classes_)
    model = ImprovedToxicModel(GAN_INPUT_DIM, EMBED_DIM, EXP_DIM, num_sp, NUM_CLASSES).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    
    f1a, f2a_raw, f3a, f4a, hb_a = extract_mol_features_raw(smi1)
    f1b, f2b_raw, f3b, f4b, hb_b = extract_mol_features_raw(smi2)
    f2a = pca.transform(f2a_raw.reshape(1, -1))[0]
    f2b = pca.transform(f2b_raw.reshape(1, -1))[0]
    assayA = np.array([a1,a2,a3], dtype=np.float32)
    assayB = np.array([a4,a5,a6], dtype=np.float32)
    inter_feat = calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b)
    
    modA = [
        torch.from_numpy(f1a).float().unsqueeze(0).to(device),
        torch.from_numpy(f2a).float().unsqueeze(0).to(device),
        torch.from_numpy(f3a).float().unsqueeze(0).to(device),
        torch.from_numpy(f4a).float().unsqueeze(0).to(device),
        torch.from_numpy(hb_a).float().unsqueeze(0).to(device),
        torch.from_numpy(assayA).float().unsqueeze(0).to(device)
    ]
    modB = [
        torch.from_numpy(f1b).float().unsqueeze(0).to(device),
        torch.from_numpy(f2b).float().unsqueeze(0).to(device),
        torch.from_numpy(f3b).float().unsqueeze(0).to(device),
        torch.from_numpy(f4b).float().unsqueeze(0).to(device),
        torch.from_numpy(hb_b).float().unsqueeze(0).to(device),
        torch.from_numpy(assayB).float().unsqueeze(0).to(device)
    ]
    with torch.no_grad():
        HA = fusion_model(modA).squeeze(0).cpu().numpy()
        HB = fusion_model(modB).squeeze(0).cpu().numpy()
    fusion_feat = np.concatenate([HA, HB, inter_feat]).reshape(1, -1)
    fusion_feat_scaled = scaler.transform(fusion_feat)[0]
    
    fusion_ad_res = ad_fusion.judge_sample(fusion_feat_scaled, f2a_raw, f2b_raw)
    
    if species_name in sp_enc.classes_:
        sp_id = sp_enc.transform([species_name])[0]
    else:
        sp_id = sp_enc.transform(["Unknown"])[0]
    exp_norm = exp_scaler.transform(np.array([[exp_day]]))[0][0]
    
    feat_tensor = torch.from_numpy(fusion_feat_scaled).float().unsqueeze(0).to(device)
    sp_tensor = torch.tensor([sp_id], dtype=torch.long).to(device)
    exp_tensor = torch.tensor([[exp_norm]], dtype=torch.float32).to(device)
    with torch.no_grad():
        logits, _ = model(feat_tensor, sp_tensor, exp_tensor)
    label_inv_map = {v:k for k,v in label_map.items()}
    
    output_dict = {
        "SMILES1": smi1,
        "SMILES2": smi2,
        "【融合AD判定】域内": fusion_ad_res["in_domain"],
        "融合AD马氏距离": fusion_ad_res["md"],
        "融合AD5-NN均值距离": fusion_ad_res["knn_avg"],
        "分子1最小Tanimoto相似度": fusion_ad_res["min_tani_A"],
        "分子2最小Tanimoto相似度": fusion_ad_res["min_tani_B"],
        "AD判定规则": f"{AD_JUDGE_MODE} | MD/KNN分位数{MD_THRESH_RATIO} | Tanimoto截断{TANIMOTO_CUTOFF}"
    }
    return output_dict

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolTransforms
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import mahalanobis, cdist
import traceback

DIM_F1 = 20
DIM_F2_RAW = 1024
DIM_F2_PCA = 128
DIM_F3 = 10
DIM_F4 = 43
DIM_HB = 2
DIM_ONE_ASSAY = 3
DIM_INTER = 14
MODAL_DIMS_SINGLE = [DIM_F1, DIM_F2_PCA, DIM_F3, DIM_F4, DIM_HB, DIM_ONE_ASSAY]
FUSION_HIDDEN = 192
HA_HB_DIM = 192
GAN_INPUT_DIM = HA_HB_DIM * 2 + DIM_INTER
EMBED_DIM = 32
EXP_DIM = 32
INPUT_DIM = GAN_INPUT_DIM + EMBED_DIM + EXP_DIM
NUM_CLASSES = 5
K_NEIGHBOR = 5
AD_JUDGE_MODE = "loose"

label_map = {
    "Antagonistic effect": 0,
    "Synergistic effect": 1,
    "Not mentioned": 2,
    "Additive effect": 3,
    "Independent effect": 4
}
label_inv_map = {v: k for k, v in label_map.items()}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_CFG = [
    {
        "name": "生殖毒性",
        "ckpt_path": "reproduction_modelAD.pth",
        "assay_cols": [
            "er_pred_hitc-SMILES1", "ar_pred_hitc-SMILES1", "cyp19_pred_hitc-SMILES1",
            "er_pred_hitc-SMILES2", "ar_pred_hitc-SMILES2", "cyp19_pred_hitc-SMILES2"
        ],
        "md_ratio": 0.98,
        "knn_ratio": 0.98,
        "tani_cut": 0.50
    },
    {
        "name": "神经毒性",
        "ckpt_path": "neurotoxicity_modelAD.pth",
        "assay_cols": [
            "neuro1_pred_hitc-SMILES1", "neuro2_pred_hitc-SMILES1", "neuro3_pred_hitc-SMILES1",
            "neuro1_pred_hitc-SMILES2", "neuro2_pred_hitc-SMILES2", "neuro3_pred_hitc-SMILES2"
        ],
        "md_ratio": 0.98,
        "knn_ratio": 0.98,
        "tani_cut": 0.50
    },
    {
        "name": "肝脏毒性",
        "ckpt_path": "hepatotoxicity_modelAD.pth",
        "assay_cols": [
            "hep1_pred_hitc-SMILES1", "hep2_pred_hitc-SMILES1", "hep3_pred_hitc-SMILES1",
            "hep1_pred_hitc-SMILES2", "hep2_pred_hitc-SMILES2", "hep3_pred_hitc-SMILES2"
        ],
        "md_ratio": 0.95,
        "knn_ratio": 0.95,
        "tani_cut": 0.60
    },
    {
        "name": "免疫毒性",
        "ckpt_path": "immunotoxicity_modelAD.pth",
        "assay_cols": [
            "imm1_pred_positive_prob-SMILES1", "imm2_pred_positive_prob-SMILES1", "imm3_pred_positive_prob-SMILES1",
            "imm1_pred_positive_prob-SMILES2", "imm2_pred_positive_prob-SMILES2", "imm3_pred_positive_prob-SMILES2"
        ],
        "md_ratio": 0.98,
        "knn_ratio": 0.98,
        "tani_cut": 0.50
    },
    {
        "name": "发育毒性",
        "ckpt_path": "developmental-toxicity_modelAD.pth",
        "assay_cols": [
            "dev1_pred_hitc-SMILES1", "dev2_pred_hitc-SMILES1", "dev3_pred_hitc-SMILES1",
            "dev1_pred_hitc-SMILES2", "dev2_pred_hitc-SMILES2", "dev3_pred_hitc-SMILES2"
        ],
        "md_ratio": 0.98,
        "knn_ratio": 0.98,
        "tani_cut": 0.50
    }
]

def to_fixed_len(arr, target_len):
    arr = np.asarray(arr, dtype=np.float32).ravel()
    if len(arr) < target_len:
        arr = np.concatenate([arr, np.zeros(target_len - len(arr), dtype=np.float32)])
    else:
        arr = arr[:target_len]
    return arr

def generate_3d_conformer(mol, max_attempts=10):
    try:
        mol_3d = Chem.AddHs(mol)
        for _ in range(max_attempts):
            if AllChem.EmbedMolecule(mol_3d, randomSeed=42) == 0:
                AllChem.MMFFOptimizeMolecule(mol_3d)
                return mol_3d
        return None
    except:
        return None

def get_3d_geometric_features(mol_3d):
    feats = []
    if mol_3d is not None and mol_3d.GetNumConformers() > 0:
        try:
            conf = mol_3d.GetConformer()
            num_atoms = mol_3d.GetNumAtoms()
            coords = np.array([[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
                               for i in range(num_atoms)])
            feats.extend(coords.mean(axis=0).tolist())
            feats.extend(coords.std(axis=0).tolist())
            feats.extend(coords.max(axis=0).tolist())
            feats.extend(coords.min(axis=0).tolist())

            bond_lengths = [rdMolTransforms.GetBondLength(conf, b.GetBeginAtomIdx(), b.GetEndAtomIdx())
                            for b in mol_3d.GetBonds()]
            feats += [np.mean(bond_lengths), np.std(bond_lengths), np.max(bond_lengths), np.min(bond_lengths)] if bond_lengths else [0.0] * 4

            angles = []
            for a in range(num_atoms):
                for n in mol_3d.GetAtomWithIdx(a).GetNeighbors():
                    angles.append(rdMolTransforms.GetAngleDeg(conf, a, n.GetIdx(), a))
            feats += [np.mean(angles), np.std(angles), np.max(angles), np.min(angles)] if angles else [0.0] * 4

            dihedrals = []
            for b in mol_3d.GetBonds():
                a1, a2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
                for n1 in mol_3d.GetAtomWithIdx(a1).GetNeighbors():
                    for n2 in mol_3d.GetAtomWithIdx(a2).GetNeighbors():
                        if n1.GetIdx() != a2 and n2.GetIdx() != a1:
                            dihedrals.append(rdMolTransforms.GetDihedralDeg(conf, n1.GetIdx(), a1, a2, n2.GetIdx()))
            feats += dihedrals[:20]
        except:
            pass
    return to_fixed_len(feats, DIM_F4)

def extract_mol_features_raw(smi):
    f1 = np.zeros(DIM_F1, dtype=np.float32)
    f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f3 = np.zeros(DIM_F3, dtype=np.float32)
    f4 = np.zeros(DIM_F4, dtype=np.float32)
    hb = np.zeros(DIM_HB, dtype=np.float32)

    if pd.isna(smi) or not isinstance(smi, str) or len(smi.strip()) == 0:
        return f1, f2_raw, f3, f4, hb

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return f1, f2_raw, f3, f4, hb

    f1 = np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumValenceElectrons(mol), Descriptors.HeavyAtomCount(mol),
        Descriptors.NHOHCount(mol), Descriptors.NOCount(mol), Descriptors.FractionCSP3(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
        Descriptors.NumSaturatedRings(mol), Descriptors.NumHeteroatoms(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.MaxPartialCharge(mol) if Descriptors.MaxPartialCharge(mol) else 0.0,
        Descriptors.MinPartialCharge(mol) if Descriptors.MinPartialCharge(mol) else 0.0,
        Descriptors.MolMR(mol), 0.0, 0.0,
    ], dtype=np.float32)
    f1 = to_fixed_len(f1, DIM_F1)

    try:
        fp_bit = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=DIM_F2_RAW)
        f2_raw = np.array(fp_bit, dtype=np.float32)
    except:
        f2_raw = np.zeros(DIM_F2_RAW, dtype=np.float32)
    f2_raw = to_fixed_len(f2_raw, DIM_F2_RAW)

    f3 = np.array([
        Descriptors.MolWt(mol) * 0.1, Descriptors.MolLogP(mol) * 0.5, Descriptors.TPSA(mol) * 0.01,
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.NumAliphaticRings(mol),
        Descriptors.NumHeteroatoms(mol), Descriptors.NumRotatableBonds(mol),
        Descriptors.LabuteASA(mol) if Descriptors.LabuteASA(mol) else 0.0,
    ], dtype=np.float32)
    f3 = to_fixed_len(f3, DIM_F3)

    mol_3d = generate_3d_conformer(mol)
    f4 = get_3d_geometric_features(mol_3d)

    h_don = Descriptors.NumHDonors(mol)
    h_acc = Descriptors.NumHAcceptors(mol)
    hb = np.array([h_don, h_acc], dtype=np.float32)
    hb = to_fixed_len(hb, DIM_HB)

    return f1, f2_raw, f3, f4, hb

def calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b):
    def cos_sim(u, v):
        n1, n2 = np.linalg.norm(u), np.linalg.norm(v)
        return 0.0 if (n1 < 1e-8 or n2 < 1e-8) else np.dot(u, v) / (n1 * n2)
    def l2_dist(u, v):
        return np.linalg.norm(u - v)

    cos1 = cos_sim(f1a, f1b)
    dist1 = l2_dist(f1a, f1b)
    had1 = (f1a * f1b).mean()

    cos3 = cos_sim(f4a, f4b)
    dist3 = l2_dist(f4a, f4b)
    had3 = (f4a * f4b).mean()

    had_low = (f3a * f3b).mean()

    da, aa = hb_a[0], hb_a[1]
    db, ab = hb_b[0], hb_b[1]
    sum_d, sum_a = da + db, aa + ab
    diff_d, diff_a = abs(da - db), abs(aa - ab)
    mul_d, mul_a = da * db, aa * ab

    inter = np.array([
        cos1, dist1, had1, cos3, dist3, had3, had_low,
        sum_d, sum_a, diff_d, diff_a, mul_d, mul_a, 0.0
    ], dtype=np.float32)
    return to_fixed_len(inter, DIM_INTER)

class CrossInteractAttnFusion(nn.Module):
    def __init__(self, modal_dims, hidden_dim, out_dim, head=4):
        super().__init__()
        self.modal_dims = modal_dims
        self.num_modal = len(modal_dims)
        self.head = head
        self.head_dim = hidden_dim // head
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim

        self.projs = nn.ModuleList([nn.Linear(d, hidden_dim) for d in modal_dims])
        self.ln_proj = nn.LayerNorm(hidden_dim)
        self.qkv_global = nn.Linear(hidden_dim, hidden_dim * 3)
        self.assay_gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Sigmoid()
        )
        self.cross_q = nn.Linear(hidden_dim, hidden_dim)
        self.cross_k = nn.Linear(hidden_dim, hidden_dim)
        self.cross_v = nn.Linear(hidden_dim, hidden_dim)
        self.ln_cross = nn.LayerNorm(hidden_dim)
        self.concat_ln = nn.LayerNorm(hidden_dim * self.num_modal)
        self.out_linear = nn.Linear(hidden_dim * self.num_modal, out_dim)
        self.drop = nn.Dropout(0.35)

    def forward(self, modal_list):
        bs = modal_list[0].shape[0]
        hidden_list = []
        for idx, m in enumerate(modal_list):
            h = self.projs[idx](m)
            h = self.ln_proj(h)
            hidden_list.append(h)
        stack = torch.stack(hidden_list, dim=1)
        qkv = self.qkv_global(stack).reshape(bs, self.num_modal, 3, self.head, self.head_dim).permute(2, 0, 3, 1, 4)
        q_g, k_g, v_g = qkv[0], qkv[1], qkv[2]
        attn_score = torch.matmul(q_g, k_g.transpose(-1, -2)) / np.sqrt(self.head_dim)
        attn_weight = F.softmax(attn_score, dim=-1)
        global_attn_out = torch.matmul(attn_weight, v_g).transpose(1, 2).reshape(bs, self.num_modal, self.hidden_dim)
        assay_feat = stack[:, -1, :]
        gate_w = self.assay_gate(assay_feat).unsqueeze(1)
        gated_feat = global_attn_out * gate_w
        molA_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        molB_feat = gated_feat[:, :5, :].mean(dim=1, keepdim=True)
        q_c = self.cross_q(molA_feat)
        k_c = self.cross_k(molB_feat)
        v_c = self.cross_v(molB_feat)
        cross_attn = F.softmax(torch.matmul(q_c, k_c.transpose(-1, -2)) / np.sqrt(self.head_dim), dim=-1)
        cross_out = torch.matmul(cross_attn, v_c)
        cross_out = self.ln_cross(cross_out)
        fused_all = gated_feat + cross_out
        concat = fused_all.reshape(bs, -1)
        concat = self.concat_ln(concat)
        out = self.drop(self.out_linear(concat))
        return out

class ModelApplicabilityDomain:
    def __init__(self, train_X_scaled, train_fpA, train_fpB, md_ratio, knn_ratio, tani_cut):
        self.X_train = train_X_scaled
        self.fpA_train = train_fpA
        self.fpB_train = train_fpB
        self.md_ratio = md_ratio
        self.knn_ratio = knn_ratio
        self.tani_cut = tani_cut
        self.n_train = self.X_train.shape[0]

        self.X_mean = np.mean(self.X_train, axis=0)
        self.X_cov = np.cov(self.X_train.T)
        self.X_cov_inv = np.linalg.pinv(self.X_cov)
        train_md = np.array([mahalanobis(x, self.X_mean, self.X_cov_inv) for x in self.X_train])
        self.md_threshold = np.quantile(train_md, self.md_ratio)

        knn_dist = cdist(self.X_train, self.X_train, metric="euclidean")
        np.fill_diagonal(knn_dist, np.inf)
        knn_sort = np.sort(knn_dist, axis=1)
        avg_k_dist = np.mean(knn_sort[:, :K_NEIGHBOR], axis=1)
        self.knn_threshold = np.quantile(avg_k_dist, self.knn_ratio)

    def calc_min_tanimoto(self, fp_new, fp_train_arr):
        fp_new_bin = fp_new.astype(bool)
        t_list = []
        for fp_t in fp_train_arr:
            fp_t_bin = fp_t.astype(bool)
            inter = np.logical_and(fp_new_bin, fp_t_bin).sum()
            union = np.logical_or(fp_new_bin, fp_t_bin).sum()
            t = inter / union if union > 1e-8 else 0.0
            t_list.append(t)
        t_list.sort(reverse=True)
        top5 = t_list[:5]
        return np.mean(top5)

    def judge_sample(self, x_scaled, fpA_new, fpB_new):
        md = mahalanobis(x_scaled, self.X_mean, self.X_cov_inv)
        md_in = md < self.md_threshold
        dist_all = cdist(x_scaled.reshape(1, -1), self.X_train, metric="euclidean")[0]
        dist_sort = np.sort(dist_all)
        avg_knn = np.mean(dist_sort[:K_NEIGHBOR])
        knn_in = avg_knn < self.knn_threshold
        tA = self.calc_min_tanimoto(fpA_new, self.fpA_train)
        tB = self.calc_min_tanimoto(fpB_new, self.fpB_train)
        fp_in = (tA >= self.tani_cut) or (tB >= self.tani_cut)
        if AD_JUDGE_MODE == "strict":
            in_domain = md_in and knn_in and fp_in
        else:
            in_domain = (md_in and knn_in) or fp_in
        return {
            "md": md,
            "md_thresh": self.md_threshold,
            "knn_avg": avg_knn,
            "knn_thresh": self.knn_threshold,
            "min_tani_A": tA,
            "min_tani_B": tB,
            "in_domain": in_domain
        }

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.4):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()
    def forward(self, x):
        residual = x
        out = self.dropout(self.act(self.bn1(self.fc1(x))))
        out = self.bn2(self.fc2(out))
        out += residual
        return self.act(out)

class ImprovedToxicModel(nn.Module):
    def __init__(self, total_feat_dim, embed_dim, exp_dim, species_num, num_classes):
        super().__init__()
        self.embed_dim = embed_dim
        self.exp_dim = exp_dim
        self.total_feat_dim = total_feat_dim
        self.species_emb = nn.Embedding(species_num, embed_dim)
        self.exp_proj = nn.Linear(1, exp_dim)
        self.backbone = nn.Sequential(
            nn.Linear(INPUT_DIM, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(1024),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.45),
            ResBlock(512),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            ResBlock(256),
        )
        self.classifier = nn.Linear(256, num_classes)
    def forward(self, full_feat, species, exposure):
        sp_feat = self.species_emb(species)
        exp_feat = F.gelu(self.exp_proj(exposure)).squeeze(1)
        all_feat = torch.cat([full_feat, sp_feat, exp_feat], dim=-1)
        hidden = self.backbone(all_feat)
        logits = self.classifier(hidden)
        return logits, hidden

def single_model_predict(ckpt_path, md_ratio, knn_ratio, tani_cut, smi1, smi2, assay_vals, species_name, exp_day):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device)
    scaler = ckpt["scaler"]
    exp_scaler = ckpt["exp_scaler"]
    pca = ckpt["pca"]
    sp_enc = ckpt["sp_enc"]
    ad_fusion_raw = ckpt["ad_fusion"]
    train_X = ad_fusion_raw.X_train
    train_fpA = ad_fusion_raw.fpA_train
    train_fpB = ad_fusion_raw.fpB_train
    ad_fusion = ModelApplicabilityDomain(train_X, train_fpA, train_fpB, md_ratio, knn_ratio, tani_cut)
    fusion_model = CrossInteractAttnFusion(MODAL_DIMS_SINGLE, FUSION_HIDDEN, HA_HB_DIM).to(device)
    if "fusion_model" in ckpt:
        fusion_model.load_state_dict(ckpt["fusion_model"])
    elif "fusion" in ckpt:
        fusion_model.load_state_dict(ckpt["fusion"])
    else:
        raise Exception(f"{ckpt_path} 无融合网络参数")
    fusion_model.eval()
    num_sp = len(sp_enc.classes_)
    model = ImprovedToxicModel(GAN_INPUT_DIM, EMBED_DIM, EXP_DIM, num_sp, NUM_CLASSES).to(device)
    if "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    elif "model" in ckpt:
        model.load_state_dict(ckpt["model"])
    else:
        raise Exception(f"{ckpt_path} 无主干分类网络参数")
    model.eval()
    f1a, f2a_raw, f3a, f4a, hb_a = extract_mol_features_raw(smi1)
    f1b, f2b_raw, f3b, f4b, hb_b = extract_mol_features_raw(smi2)
    f2a = pca.transform(f2a_raw.reshape(1, -1))[0]
    f2b = pca.transform(f2b_raw.reshape(1, -1))[0]
    assayA = np.array(assay_vals[:3], dtype=np.float32)
    assayB = np.array(assay_vals[3:], dtype=np.float32)
    inter_feat = calc_mol_mixed_interaction(f1a, f1b, f3a, f3b, f4a, f4b, hb_a, hb_b)
    modA = [
        torch.from_numpy(f1a).float().unsqueeze(0).to(device),
        torch.from_numpy(f2a).float().unsqueeze(0).to(device),
        torch.from_numpy(f3a).float().unsqueeze(0).to(device),
        torch.from_numpy(f4a).float().unsqueeze(0).to(device),
        torch.from_numpy(hb_a).float().unsqueeze(0).to(device),
        torch.from_numpy(assayA).float().unsqueeze(0).to(device)
    ]
    modB = [
        torch.from_numpy(f1b).float().unsqueeze(0).to(device),
        torch.from_numpy(f2b).float().unsqueeze(0).to(device),
        torch.from_numpy(f3b).float().unsqueeze(0).to(device),
        torch.from_numpy(f4b).float().unsqueeze(0).to(device),
        torch.from_numpy(hb_b).float().unsqueeze(0).to(device),
        torch.from_numpy(assayB).float().unsqueeze(0).to(device)
    ]
    with torch.no_grad():
        HA = fusion_model(modA).squeeze(0).cpu().numpy()
        HB = fusion_model(modB).squeeze(0).cpu().numpy()
    fusion_feat = np.concatenate([HA, HB, inter_feat]).reshape(1, -1)
    fusion_feat_scaled = scaler.transform(fusion_feat)[0]
    ad_res = ad_fusion.judge_sample(fusion_feat_scaled, f2a_raw, f2b_raw)
    if species_name in sp_enc.classes_:
        sp_id = sp_enc.transform([species_name])[0]
    else:
        sp_id = sp_enc.transform(["Unknown"])[0]
    exp_norm = exp_scaler.transform(np.array([[exp_day]]))[0][0]
    feat_tensor = torch.from_numpy(fusion_feat_scaled).float().unsqueeze(0).to(device)
    sp_tensor = torch.tensor([sp_id], dtype=torch.long).to(device)
    exp_tensor = torch.tensor([[exp_norm]], dtype=torch.float32).to(device)
    with torch.no_grad():
        logits, _ = model(feat_tensor, sp_tensor, exp_tensor)
    out = {
        "AD_in_domain": ad_res["in_domain"],
        "MD": ad_res["md"],
        "MD_thresh": ad_res["md_thresh"],
        "KNN_avg": ad_res["knn_avg"],
        "KNN_thresh": ad_res["knn_thresh"],
        "min_tani_A": ad_res["min_tani_A"],
        "min_tani_B": ad_res["min_tani_B"]
    }
    return out

excel_path = "chemical-to-predict.xlsx"
output_csv = "five_toxicity_all_pred_resultAD.csv"
df_input = pd.read_excel(excel_path)
print(df_input.columns.tolist())
all_result = []
for row_idx, row in df_input.iterrows():
    print(f"\n===== 正在预测第{row_idx}行 =====")
    row_base = {
        "SMILES1": row["SMILES1"],
        "SMILES2": row["SMILES2"],
        "species": row["species"],
        "exposure_day": row["exposure duration (days)"]
    }
    row_final = row_base.copy()
    for cfg in MODEL_CFG:
        m_name = cfg["name"]
        ckpt = cfg["ckpt_path"]
        assay_cols = cfg["assay_cols"]
        md_r = cfg["md_ratio"]
        knn_r = cfg["knn_ratio"]
        tani_c = cfg["tani_cut"]
        try:
            assay_vals = [float(row[c]) if not pd.isna(row[c]) else 0.0 for c in assay_cols]
            pred_res = single_model_predict(
                ckpt_path=ckpt,
                md_ratio=md_r,
                knn_ratio=knn_r,
                tani_cut=tani_c,
                smi1=row["SMILES1"],
                smi2=row["SMILES2"],
                assay_vals=assay_vals,
                species_name=str(row["species"]),
                exp_day=float(row["exposure duration (days)"]) if not pd.isna(row["exposure duration (days)"]) else 1.0
            )
            row_final[f"{m_name}_AD是否域内"] = pred_res["AD_in_domain"]
            row_final[f"{m_name}_融合AD马氏距离"] = pred_res["MD"]
            row_final[f"{m_name}_融合AD马氏阈值"] = pred_res["MD_thresh"]
            row_final[f"{m_name}_融合AD5-NN平均距离"] = pred_res["KNN_avg"]
            row_final[f"{m_name}_融合AD5-NN阈值"] = pred_res["KNN_thresh"]
            row_final[f"{m_name}_分子1最小Tanimoto相似度"] = pred_res["min_tani_A"]
            row_final[f"{m_name}_分子2最小Tanimoto相似度"] = pred_res["min_tani_B"]
            print(f"✅ {m_name} 预测成功")
        except Exception as e:
            traceback.print_exc()
            print(f"❌ {m_name} 预测失败，报错：{str(e)}")
            row_final[f"{m_name}_AD是否域内"] = None
            row_final[f"{m_name}_融合AD马氏距离"] = None
            row_final[f"{m_name}_融合AD马氏阈值"] = None
            row_final[f"{m_name}_融合AD5-NN平均距离"] = None
            row_final[f"{m_name}_融合AD5-NN阈值"] = None
            row_final[f"{m_name}_分子1最小Tanimoto相似度"] = None
            row_final[f"{m_name}_分子2最小Tanimoto相似度"] = None
    all_result.append(row_final)
df_out = pd.DataFrame(all_result)
df_out.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"\n批量预测完成！共{len(result_list)}条，结果已保存 {output_csv}")
in_domain_cnt = df_out[[col for col in df_out.columns if "AD是否域内" in col]].sum(axis=0)
print("各毒性域内样本统计：")
print(in_domain_cnt)